In [10]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langgraph.graph import StateGraph,START,END
from langchain_core.prompts import ChatPromptTemplate

In [2]:
if os.environ.get("OPENAI_API_KEY"):
    API_KEY = os.environ.get("OPENAI_API_KEY")
    print("api key is set!")
else:
    raise ValueError("print api key doesn't ecists!")

api key is set!


In [4]:
llm = init_chat_model("gpt-5-nano",temperature=0)

In [ ]:
my_promts = ChatPromptTemplate.from_messages([
    ("system","")
])

In [3]:
from typing import TypedDict,List
class graph_schema(TypedDict):
    message_manual:list
    

In [5]:
def create_post(state:graph_schema)->graph_schema:
    message_manual = state['message_manual']
    response_manual = llm.invoke(message_manual).content
    ai_response_manual = AIMessage(content = response_manual)
    state['message_manual'] = message_manual + [ai_response_manual]
    return state

def curated_post(state:graph_schema)->graph_schema:
    message_manual = state['message_manual']
    response_manual = llm.invoke(message_manual).content
    ai_response_manual = AIMessage(content = response_manual)
    state['message_manual'] = message_manual + [ai_response_manual]
    return state


In [6]:
graph = StateGraph(graph_schema)

graph.add_node("create_post",create_post)
graph.add_node("curated_post",curated_post)

graph.add_edge(START,'create_post')
graph.add_edge('create_post','curated_post')
graph.add_edge('curated_post',END)

In [7]:
message_graph = graph.compile()

In [8]:
message_graph.invoke({"message_manual":[HumanMessage(content= "When was Donald Trump become president of USA?")]})

{'message_manual': [HumanMessage(content='When was Donald Trump become president of USA?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Donald Trump was inaugurated as the 45th president of the United States on January 20, 2017. His term lasted until January 20, 2021.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  AIMessage(content='He was inaugurated as the 45th president on January 20, 2017, and served until January 20, 2021.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}